In [199]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [200]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from typing import List, Tuple

# 1. Create the dataset (simulated)

In [201]:
timestamps: List[int] = [0, 4, 7, 20, 23, 25, 30, 32]
block_size: int = 10
threshold: float = 0.5

In [202]:
heatmaps: List[np.ndarray] = [np.random.rand(100, 100) for _ in range(len(timestamps))]

In [203]:
def extract_features(heatmap: np.ndarray, timestamp: int) -> list[list[float]]:
    blocks: list[list[float]] = []

    for i in range(0, heatmap.shape[0], block_size):
        for j in range(0, heatmap.shape[1], block_size):
            block = heatmap[i:i+block_size, j:j+block_size]
            mean_intensity: float = float(np.mean(block))
            max_intensity: float = float(np.max(block))
            min_intensity: float = float(np.min(block))
            var_intensity: float = float(np.var(block))
            
            blocks.append([mean_intensity, max_intensity, min_intensity, var_intensity, float(timestamp)])
    
    return blocks

# 2. Train the model

In [204]:
all_features: List[List[float]] = []
for heatmap, timestamp in zip(heatmaps, timestamps):
    blocks = extract_features(heatmap, timestamp)
    all_features.extend(blocks)

features_df: pd.DataFrame = pd.DataFrame(
    all_features, 
    columns=['mean', 'max', 'min', 'variance', 'timestamp']
)

interval_labels: List[int] = []

for i in range(1, len(timestamps)):
    prev_time: int = timestamps[i - 1]
    curr_time: int = timestamps[i]
    interval: int = (curr_time - prev_time) // 2
    interval_labels.append(interval)

blocks_per_heatmap: int = heatmaps[0].shape[0] // block_size * (heatmaps[0].shape[1] // block_size)

expanded_labels: List[int] = []
for interval in interval_labels:
    expanded_labels.extend([interval] * blocks_per_heatmap)

features_df = features_df[blocks_per_heatmap:]
features_df['time_interval'] = expanded_labels

X: pd.DataFrame = features_df[['mean', 'max', 'min', 'variance', 'timestamp']]
y: pd.Series = features_df['time_interval']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [205]:
X: pd.DataFrame = features_df[['mean', 'max', 'min', 'variance', 'timestamp']]
y: pd.Series = features_df['time_interval']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model: KNeighborsClassifier = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)

y_pred: np.ndarray = model.predict(X_test)
accuracy: float = accuracy_score(y_test, y_pred)
print(f"Acurácia do modelo: {accuracy * 100:.2f}%")

Acurácia do modelo: 28.57%


# 3. Check global occurrences in the dataset (optional, debug)

In [229]:
def find_first_interval_above_threshold(heatmaps: list[np.ndarray], timestamps: list[int], threshold: float = 0.5) -> int:
    """Retorna o índice do primeiro intervalo em que a média global do heatmap ultrapassa o limiar."""
    for idx, (heatmap, ts) in enumerate(zip(heatmaps, timestamps)):
        global_mean = np.mean(heatmap)
        if global_mean > threshold:
            print(f"Incidência global ultrapassou {threshold} no intervalo de timestamp {ts} (índice {idx}) com média {global_mean:.3f}")
    print(f"Nenhum intervalo ultrapassou o limiar de {threshold}.")
    return -1

first_idx = find_first_interval_above_threshold(heatmaps, timestamps, threshold)

Incidência global ultrapassou 0.5 no intervalo de timestamp 0 (índice 0) com média 0.501
Incidência global ultrapassou 0.5 no intervalo de timestamp 20 (índice 3) com média 0.503
Incidência global ultrapassou 0.5 no intervalo de timestamp 25 (índice 5) com média 0.500
Nenhum intervalo ultrapassou o limiar de 0.5.


# 4. Predict the interval between global occurrences

In [228]:
def predict_next_spike_period(heatmaps: list[np.ndarray], timestamps: list[int], model: KNeighborsClassifier, threshold: float = 0.5) -> int | None:
    spike_indices: list[int] = [i for i, h in enumerate(heatmaps) if np.mean(h) > threshold]
    if len(spike_indices) < 2:
        print("Menos de duas ocorrências acima do limiar. Não é possível calcular o período.")
        return None
    
    periods: list[int] = [timestamps[spike_indices[i]] - timestamps[spike_indices[i-1]] for i in range(1, len(spike_indices))]
    print(f'timestamps: {timestamps}')
    print(f"Períodos reais entre ocorrências: {periods}")
    
    last_heatmap: np.ndarray = heatmaps[spike_indices[-1]]
    last_timestamp: int = timestamps[spike_indices[-1]]
    features: list[list[float]] = extract_features(last_heatmap, last_timestamp)
    features_df: pd.DataFrame = pd.DataFrame(features, columns=['mean', 'max', 'min', 'variance', 'timestamp'])
    pred: np.ndarray = model.predict(features_df)
    predicted_period: int = int(np.round(np.mean(pred)))
    print(f"Período previsto até a próxima ocorrência global acima de {threshold}: {predicted_period}")
    return predicted_period

predict_next_spike_period(heatmaps, timestamps, model, threshold)

timestamps: [0, 4, 7, 20, 23, 25, 30, 32]
Períodos reais entre ocorrências: [20, 5]
Período previsto até a próxima ocorrência global acima de 0.5: 1


1